In [ ]:
# Cell 1: Setup for Ben-Moshe dataset

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from importlib.metadata import version

# Scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(8, 6))

# Paths
from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_raw = os.path.join(base_path, "Data/raw")
data_processed = os.path.join(base_path, "Data/processed")
results_path = os.path.join(base_path, "results")

benmoshe_path = os.path.join(data_raw, "benmoshe_zenodo6035873")

print("✅ Setup complete for Ben-Moshe dataset!")
print(f"Scanpy version: {version('scanpy')}")

# List files
print("\n📁 Files in Ben-Moshe directory:")
if os.path.exists(benmoshe_path):
    files = os.listdir(benmoshe_path)
    for f in sorted(files):
        if not f.startswith('.'):
            file_path = os.path.join(benmoshe_path, f)
            if os.path.isfile(file_path):
                size = os.path.getsize(file_path) / 1e9
                print(f"   {f} ({size:.2f} GB)")
else:
    print(f"   ⚠️  Directory not found: {benmoshe_path}")

In [ ]:
# Cell 2: Load Ben-Moshe metadata (FINAL CORRECTION)

metadata_file = os.path.join(benmoshe_path, "Single_cell_Meta_data.txt")

print("📂 Loading metadata...")

# Load with proper header
metadata = pd.read_csv(metadata_file, sep=',')  # It's comma-separated, not tab!

print(f"✅ Metadata loaded!")
print(f"   Rows (cells): {metadata.shape[0]:,}")
print(f"   Columns: {metadata.shape[1]}")

print("\n📋 Column names:")
print(metadata.columns.tolist())

print("\n📊 First few rows:")
print(metadata.head(10))

# Set cell barcode as index
metadata.set_index('Cell_barcode', inplace=True)

print("\n📊 VALUE COUNTS:")

print("\n🔬 Cell types:")
print(metadata['cell_type'].value_counts())

print("\n⏱️ Timepoints:")
print(metadata['time_point'].value_counts().sort_index())

print("\n🧪 Run IDs (samples):")
print(metadata['run_id'].value_counts())

print("\n🗺️ Zones:")
print(metadata['zone'].value_counts().sort_index())

print("\n✅ Metadata ready for integration with count matrix!")

In [ ]:
# Cell 3: Load Ben-Moshe count matrix (CORRECTED)

counts_file = os.path.join(benmoshe_path, "Single_cell_UMI_COUNT.txt")

print("📂 Loading count matrix (2 GB file - will take 2-3 minutes)...")

# Load with comma separator and proper header
counts = pd.read_csv(counts_file, sep=',', index_col=0)

print(f"\n✅ Count matrix loaded!")
print(f"   Genes (rows): {counts.shape[0]:,}")
print(f"   Cells (columns): {counts.shape[1]:,}")

print("\n📊 First 5 genes × 5 cells:")
print(counts.iloc[:5, :5])

print("\n📊 Matrix statistics:")
total_umis = counts.sum().sum()
print(f"   Total UMIs: {total_umis:,.0f}")
print(f"   Mean UMIs/cell: {counts.sum(axis=0).mean():.0f}")
print(f"   Median UMIs/cell: {counts.sum(axis=0).median():.0f}")
print(f"   Mean genes detected/cell: {(counts > 0).sum(axis=0).mean():.0f}")

print("\n✅ Count matrix ready!")

In [ ]:
# Cell 4: Create AnnData object

# Transpose (genes × cells → cells × genes)
print("🔄 Transposing count matrix...")
counts_T = counts.T

# Create AnnData
print("🔨 Creating AnnData object...")
adata_benmoshe = sc.AnnData(X=counts_T)

# Match and add metadata
print("📋 Matching metadata to cells...")
common_cells = adata_benmoshe.obs_names.intersection(metadata.index)
print(f"   Cells in both count matrix and metadata: {len(common_cells):,}")

adata_benmoshe = adata_benmoshe[common_cells, :].copy()
adata_benmoshe.obs = metadata.loc[common_cells]

print(f"\n✅ AnnData created!")
print(f"   Cells: {adata_benmoshe.n_obs:,}")
print(f"   Genes: {adata_benmoshe.n_vars:,}")

print("\n📊 Cells by cell type:")
print(adata_benmoshe.obs['cell_type'].value_counts())

print("\n📊 Cells by timepoint:")
print(adata_benmoshe.obs['time_point'].value_counts().sort_index())

In [ ]:
# Cell 5: Filter for hepatocytes only

print("📊 BEFORE FILTERING:")
print(f"   Total cells: {adata_benmoshe.n_obs:,}")
print(f"   Cell types: {adata_benmoshe.obs['cell_type'].nunique()}")

# Filter for hepatocytes
adata_benmoshe_hep = adata_benmoshe[adata_benmoshe.obs['cell_type'] == 'Hep'].copy()

print(f"\n📊 AFTER FILTERING (Hepatocytes only):")
print(f"   Hepatocytes: {adata_benmoshe_hep.n_obs:,}")

print("\n📊 Hepatocytes by timepoint:")
print(adata_benmoshe_hep.obs['time_point'].value_counts().sort_index())

print("\n📊 Hepatocytes by sample:")
print(adata_benmoshe_hep.obs['run_id'].value_counts())

# Replace main object
adata_benmoshe = adata_benmoshe_hep.copy()

print("\n✅ Filtered to hepatocytes only!")

In [ ]:
# Cell 6: Calculate QC metrics

# Identify mitochondrial genes
adata_benmoshe.var['mt'] = adata_benmoshe.var_names.str.startswith(('MT-', 'mt-', 'Mt-'))
n_mt = adata_benmoshe.var['mt'].sum()
print(f"🔍 Found {n_mt} mitochondrial genes")

# Calculate QC
sc.pp.calculate_qc_metrics(
    adata_benmoshe,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("\n📊 QC METRICS:")
print("="*80)
print(f"📈 Median genes/cell: {adata_benmoshe.obs['n_genes_by_counts'].median():.0f}")
print(f"📈 Median UMI/cell: {adata_benmoshe.obs['total_counts'].median():.0f}")
print(f"📈 Median mito %: {adata_benmoshe.obs['pct_counts_mt'].median():.2f}%")

print("\n📊 BY TIMEPOINT:")
for tp in sorted(adata_benmoshe.obs['time_point'].unique()):
    subset = adata_benmoshe.obs[adata_benmoshe.obs['time_point'] == tp]
    print(f"\n{tp}h: {len(subset):,} cells")
    print(f"  Median genes: {subset['n_genes_by_counts'].median():.0f}")
    print(f"  Median UMI: {subset['total_counts'].median():.0f}")
    print(f"  Median mito%: {subset['pct_counts_mt'].median():.2f}%")

print("\n✅ QC calculated!")

In [ ]:
# Cell 7: Save Ben-Moshe dataset

output_file = os.path.join(data_processed, "benmoshe_hepatocytes.h5ad")
adata_benmoshe.write(output_file)

print(f"✅ Dataset saved: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1e6:.1f} MB")

print(f"\n📊 FINAL DATASET:")
print("="*80)
print(f"   Cells: {adata_benmoshe.n_obs:,}")
print(f"   Genes: {adata_benmoshe.n_vars:,}")
print(f"   Timepoints: {sorted(adata_benmoshe.obs['time_point'].unique())}")
print(f"\n   Median genes/cell: {adata_benmoshe.obs['n_genes_by_counts'].median():.0f}")
print(f"   Median UMI/cell: {adata_benmoshe.obs['total_counts'].median():.0f}")
print("="*80)

In [ ]:
# Cell 8: Create summary table

summary_data = []

for tp in sorted(adata_benmoshe.obs['time_point'].unique()):
    subset = adata_benmoshe.obs[adata_benmoshe.obs['time_point'] == tp]
    
    summary_data.append({
        'Dataset': 'Ben-Moshe',
        'Timepoint_Hours': int(tp),
        'Cells': len(subset),
        'Samples': subset['run_id'].nunique(),
        'Median_Genes': int(subset['n_genes_by_counts'].median()),
        'Median_UMI': int(subset['total_counts'].median()),
        'Median_Mito_Pct': f"{subset['pct_counts_mt'].median():.2f}"
    })

summary_df = pd.DataFrame(summary_data)

print("📋 BEN-MOSHE SUMMARY:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

summary_file = os.path.join(data_processed, "benmoshe_summary.csv")
summary_df.to_csv(summary_file, index=False)
print(f"\n✅ Summary saved: {summary_file}")

In [ ]:
# Cell 9: Visualize Ben-Moshe data

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Cells per timepoint
tp_counts = adata_benmoshe.obs['time_point'].value_counts().sort_index()
axes[0, 0].bar(range(len(tp_counts)), tp_counts.values, color='#3498db', width=0.6)
axes[0, 0].set_xticks(range(len(tp_counts)))
axes[0, 0].set_xticklabels([f'{int(t)}h' for t in tp_counts.index])
axes[0, 0].set_ylabel('Cells', fontweight='bold')
axes[0, 0].set_title('Hepatocytes per Timepoint', fontweight='bold', fontsize=12)
for i, val in enumerate(tp_counts.values):
    axes[0, 0].text(i, val + 20, f'{val:,}', ha='center', fontsize=9, fontweight='bold')

# Plot 2: Genes per cell
axes[0, 1].hist(adata_benmoshe.obs['n_genes_by_counts'], bins=100, color='#9b59b6', alpha=0.7)
axes[0, 1].axvline(adata_benmoshe.obs['n_genes_by_counts'].median(), 
                   color='red', linestyle='--', linewidth=2,
                   label=f"Median: {adata_benmoshe.obs['n_genes_by_counts'].median():.0f}")
axes[0, 1].set_xlabel('Genes per Cell', fontweight='bold')
axes[0, 1].set_ylabel('Cells', fontweight='bold')
axes[0, 1].set_title('Genes per Cell', fontweight='bold', fontsize=12)
axes[0, 1].legend()

# Plot 3: UMI per cell
axes[1, 0].hist(adata_benmoshe.obs['total_counts'], bins=100, color='#e74c3c', alpha=0.7)
axes[1, 0].axvline(adata_benmoshe.obs['total_counts'].median(), 
                   color='red', linestyle='--', linewidth=2,
                   label=f"Median: {adata_benmoshe.obs['total_counts'].median():.0f}")
axes[1, 0].set_xlabel('UMI per Cell', fontweight='bold')
axes[1, 0].set_ylabel('Cells', fontweight='bold')
axes[1, 0].set_title('UMI per Cell', fontweight='bold', fontsize=12)
axes[1, 0].legend()

# Plot 4: Genes vs UMI
axes[1, 1].scatter(adata_benmoshe.obs['total_counts'], 
                   adata_benmoshe.obs['n_genes_by_counts'],
                   s=2, alpha=0.3, c='#2ecc71')
axes[1, 1].set_xlabel('UMI per Cell', fontweight='bold')
axes[1, 1].set_ylabel('Genes per Cell', fontweight='bold')
axes[1, 1].set_title('Genes vs UMI', fontweight='bold', fontsize=12)

plt.tight_layout()

fig_path = os.path.join(results_path, 'figures/benmoshe_hepatocytes_qc.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Figure saved: {fig_path}")